In [1]:
import pandas as pd 
import numpy as np

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
data=pd.read_csv("mentalhealthdataset.csv")

In [7]:
data['self_employed'].value_counts(dropna=False)

self_employed
No     257994
Yes     29168
NaN      5202
Name: count, dtype: int64

In [9]:
data['self_employed']=data['self_employed'].fillna('No')

In [11]:
data = data.drop_duplicates()

In [13]:
def define_risk(row):
    # 🔴 High risk — strong indicators of mental distress
    if (
        row['treatment'] == 'Yes' or
        (row['Mental_Health_History'] == 'Yes' and row['Coping_Struggles'] == 'Yes') or
        (row['Growing_Stress'] == 'Yes' and row['Work_Interest'] == 'No') or
        (row['Social_Weakness'] == 'Yes' and row['Coping_Struggles'] == 'Yes')
    ):
        return 'High'
    
    # 🟠 Medium risk — moderate indicators, some early signs
    if (
        (row['Growing_Stress'] == 'Yes') or
        (row['Coping_Struggles'] == 'Yes') or
        (row['Changes_Habits'] == 'Yes') or
        (row['Social_Weakness'] == 'Maybe') or
        (row['Work_Interest'] == 'No' and row['Mental_Health_History'] == 'No')
    ):
        return 'Medium'
    
    # 🟢 Low risk — stable behavior, coping well
    return 'Low'

# Apply and inspect distribution
data['Risk_Level'] = data.apply(define_risk, axis=1)
print(data['Risk_Level'].value_counts())


Risk_Level
High      195553
Medium     78586
Low        15912
Name: count, dtype: int64


In [15]:
from sklearn.preprocessing import LabelEncoder
binary_cols=['Gender','self_employed','family_history','treatment','Coping_Struggles']
for col in binary_cols:
    data[col]=LabelEncoder().fit_transform(data[col])

In [17]:
data['Days_Indoors']= data['Days_Indoors'].map({'Go out Every day': 0, '1-14 days': 1, '15-30 days': 2, '31-60 days': 3, 'More than 2 months': 4})

In [19]:
data['Growing_Stress'] = data['Growing_Stress'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Changes_Habits'] = data['Changes_Habits'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Mental_Health_History'] = data['Mental_Health_History'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Mood_Swings'] = data['Mood_Swings'].map({'Low': 0, 'Medium': 1, 'High': 2})
data['Work_Interest'] = data['Work_Interest'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['Social_Weakness'] = data['Social_Weakness'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['mental_health_interview'] = data['mental_health_interview'].map({'No': 0, 'Maybe': 1, 'Yes': 2})
data['care_options'] = data['care_options'].map({'No': 0, 'Maybe': 1, 'Yes': 2, 'Not sure': 1})

In [21]:
data = pd.get_dummies(data, columns=['Country', 'Occupation'], drop_first=True)

In [23]:
bool_cols = data.select_dtypes('bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [27]:
data=data.drop(columns=['Timestamp'])
data.head()

,Gender,self_employed,family_history,treatment,Days_Indoors,Growing_Stress,Changes_Habits,Mental_Health_History,Mood_Swings,Coping_Struggles,...,Country_South Africa,Country_Sweden,Country_Switzerland,Country_Thailand,Country_United Kingdom,Country_United States,Occupation_Corporate,Occupation_Housewife,Occupation_Others,Occupation_Student
0,0,0,0,1,1,2,0,2,1,0,...,0,0,0,0,0,1,1,0,0,0
1,0,0,1,1,1,2,0,2,1,0,...,0,0,0,0,0,1,1,0,0,0
2,0,0,1,1,1,2,0,2,1,0,...,0,0,0,0,0,1,1,0,0,0
3,0,0,1,1,1,2,0,2,1,0,...,0,0,0,0,0,1,1,0,0,0
4,0,0,1,1,1,2,0,2,1,0,...,0,0,0,0,0,1,1,0,0,0


In [33]:
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


X = data.drop(columns=['Risk_Level', 'treatment',])
y = data['Risk_Level']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)







gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42)


gb_model.fit(X_train, y_train)


y_pred = gb_model.predict(X_test)

print("\n--- Gradient Boosting Classification Report ---\n", classification_report(y_test, y_pred))
print("Gradient Boosting Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



--- Gradient Boosting Classification Report ---
               precision    recall  f1-score   support

        High       0.87      0.91      0.89     58666
         Low       0.80      0.71      0.75      4774
      Medium       0.80      0.71      0.75     23576

    accuracy                           0.85     87016
   macro avg       0.82      0.78      0.80     87016
weighted avg       0.85      0.85      0.85     87016

Gradient Boosting Confusion Matrix:
 [[53599   875  4192]
 [ 1368  3406     0]
 [ 6803     0 16773]]


In [37]:
# --- Step 1: Keep original categorical values for misclassified checking ---
categorical_cols = ['Gender','self_employed','family_history','treatment','Coping_Struggles',
                    'Days_Indoors','Growing_Stress','Changes_Habits','Mental_Health_History',
                    'Mood_Swings','Work_Interest','Social_Weakness','mental_health_interview',
                    'care_options']

# Use original rows corresponding to the test set
X_test_orig = data.loc[X_test.index, categorical_cols].copy()
X_test_orig['True_Risk'] = y_test
X_test_orig['Predicted_Risk'] = y_pred

# --- Step 2: Identify misclassified samples ---
misclassified = X_test_orig[X_test_orig['True_Risk'] != X_test_orig['Predicted_Risk']]

# --- Step 3: Display a few misclassified rows ---
misclassified.head(10)


,Gender,self_employed,family_history,treatment,Coping_Struggles,Days_Indoors,Growing_Stress,Changes_Habits,Mental_Health_History,Mood_Swings,Work_Interest,Social_Weakness,mental_health_interview,care_options,True_Risk,Predicted_Risk
65505,1,0,0,1,0,4,1,2,0,2,2,0,1,0,High,Medium
285541,1,0,0,1,0,2,1,0,2,1,0,1,1,0,High,Medium
144072,1,0,0,1,0,0,1,1,0,2,0,1,0,0,High,Medium
61948,1,0,1,0,0,2,2,1,0,1,1,1,0,0,Medium,High
160200,1,1,0,0,0,3,1,0,1,1,0,1,0,2,Medium,High
176150,1,0,0,1,0,1,2,1,0,0,1,0,0,0,High,Medium
62577,1,0,1,0,0,4,0,1,0,2,2,2,0,0,Low,High
38001,0,0,0,0,0,4,1,0,0,1,0,1,0,0,Medium,High
80248,1,0,0,0,0,0,0,2,2,1,0,1,0,2,Medium,High
259823,1,0,1,0,1,1,0,2,0,2,2,1,0,1,Medium,High
